# Single-fiber DM piston scans across mount focus

Analysis only: this notebook never imports acquisition code or connects to hardware.
The companion terminal script saves one spectrum per `[focus, segment, piston]`.
Spectra already average frames, subtract dark and **sum rows**; broadband flux is
`SPECTRA.sum(axis=-1)` in ADU/frame. OPD is **2 x absolute commanded DM stroke**, in um.
Focus is absolute mount Z in controller pulses, not nanometres.

The numbered 37-hexagon layout and sine-squared fringe model follow the supplied
`null_scan_parasitic_interference_analysis.ipynb`. Here there is one dispersed output:
one visibility-coloured DM map per focus, plus an MP4 with the same colour scale.
The segment inspector shows focus versus OPD, a selected fringe fit, and visibility
versus focus. Missing/rejected fits are grey; a sufficiently flat positive trace has V=0.

Run the acquisition from the repository root:
```powershell
python Matteo/Fiber_DM_piston_scans/fiber_DMfocus_piston_scan.py
```
Set `SEGMENT_PTT`, `PISTON_START/END/STEP`, `NFRAMES` and `FOCUS_START/END/STEP`
in that script. Unset focus values prompt in the terminal. Both grid endpoints
are included; the interval must be an integer multiple of the requested step.
Check the plan with `--check-only --focus-start <Z0> --focus-end <Z1> --focus-step <dZ>`.
Outputs go to the configured alignment root under
`YYYYMMDD/FIber_DMfocus_piston_scan/fiber_dm_piston_N.fits` plus a JSON sidecar.

Select that FITS in the first code cell here and run all cells. Analysis outputs
include fit results CSV, settings JSON, one DM PNG per focus, 37 per-segment
focus/OPD diagnostic PNGs, and `visibility_vs_focus.mp4`.

The notebook uses the same Windows `OUTPUT_ROOT` as the fiber-focus and fiber-piston
notebooks. It reads only `OUTPUT_ROOT/YYYYMMDD/FIber_DMfocus_piston_scan/`.
For example: `OUTPUT_ROOT/20261124/FIber_DMfocus_piston_scan/fiber_dm_piston_1.fits`.
Set `SCAN_DATE="20261124"` to select that day's folder, or leave it `None` to choose
the newest scan across dates. `DATA_FILE` can explicitly select a full path or a path
relative to `OUTPUT_ROOT`. On the instrument, acquisition uses the same configured
alignment data root as the existing focus/piston scripts, normally
`/home/scexao/glint/glintdata/alignment_scan`.


In [ ]:
# USER CONTROLS - edit this standalone cell first
from pathlib import Path

# Same Windows data-copy root as the fiber-focus and fiber-piston notebooks.
OUTPUT_ROOT = Path(r"C:\Users\mcol0021\OneDrive - The University of Sydney (Staff)\Desktop\GLINT_test reports\Fiber_scans")
SCAN_DATE = None  # e.g. "20261124"; None searches all date folders
SCAN_FOLDER = "FIber_DMfocus_piston_scan"
DATA_FILE = None  # None selects newest scan; or full path / path relative to OUTPUT_ROOT
USE_SYNTHETIC_DEMO = False  # explicitly enable to preview without acquired data
OUTPUT_DIR = None  # None -> beside DATA_FILE / <scan_name>_analysis

# Fit model: F(OPD) = c0 - A*sin(k*(OPD-mean(OPD)) + phi)**2
# V = (Fmax-Fmin)/(Fmax+Fmin) = A/(2*c0-A).
OPD_PERIOD_BOUNDS_UM = None  # None: reference notebook range [OPD_span/1.8, OPD_span/0.6]
# Can instead constrain the calibrated fringe period, e.g. (1.2, 2.0) in OPD um.
N_FREQUENCY_TRIALS = 400
MIN_POINTS_FOR_FIT = 8
FLAT_FRACTION_THRESHOLD = 1e-3
MIN_FIT_R2 = 0.35
MAX_VISIBILITY = 1.0

VISIBILITY_CMAP = "inferno"
VISIBILITY_LIMITS = (0.0, 1.0)  # shared by every focus / movie frame
VISIBILITY_LOG_SCALE = False  # if True, requires positive lower limit, e.g. (0.005, 1.0)
USE_INTERACTIVE_WIDGETS = True  # False: one static plot per tool instead of widgets
SELECTED_SEGMENT = 0
SELECTED_FOCUS_INDEX = 0
NORMALIZE_FOCUS_OPD_ROWS = False  # divide each focus's flux trace by its mean for display only

SAVE_FOCUS_PNGS = True
SAVE_SEGMENT_PNGS = True  # exports all 37 focus-versus-OPD + visibility diagnostic figures
EXPORT_MP4 = True
MOVIE_FPS = 2
MOVIE_HOLD_FRAMES = 2  # repeated frames per focus; focus order follows acquisition


In [ ]:
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits
from matplotlib.patches import Polygon
from matplotlib.collections import PatchCollection
from matplotlib.colors import Normalize, LogNorm
from matplotlib.animation import FFMpegWriter
from IPython.display import display, Video


def display_analysis_figure(fig):
    """Display one PNG and close the figure, independent of the kernel's backend."""
    import io
    from IPython.display import Image
    try:
        with plt.ioff():
            with io.BytesIO() as buffer:
                fig.savefig(buffer, format="png", dpi=fig.dpi, facecolor="white")
                display(Image(data=buffer.getvalue()))
    finally:
        plt.close(fig)


def find_scan():
    root = Path(OUTPUT_ROOT).expanduser()
    date_folder = "*" if SCAN_DATE is None else str(SCAN_DATE)
    if SCAN_DATE is not None and (len(date_folder) != 8 or not date_folder.isdigit()):
        raise ValueError("SCAN_DATE must be None or a YYYYMMDD date, e.g. '20261124'")
    pattern = f"{date_folder}/{SCAN_FOLDER}/fiber_dm_piston_*.fits"
    print("Acquisition data pattern:", root/pattern)
    files = list(root.glob(pattern))
    if not files:
        raise FileNotFoundError(f"No scan FITS matching {root/pattern}. "
                                "Set OUTPUT_ROOT to the common fiber-scan data root, SCAN_DATE to a date, "
                                "or DATA_FILE to a specific FITS. Synthetic previews require USE_SYNTHETIC_DEMO=True.")
    return max(files, key=lambda p: p.stat().st_mtime)


def selected_scan_path():
    if DATA_FILE is None:
        return find_scan().resolve()
    path = Path(DATA_FILE).expanduser()
    return (path if path.is_absolute() else Path(OUTPUT_ROOT).expanduser()/path).resolve()


def load_scan(path):
    with fits.open(path, memmap=False, checksum=True) as hdul:
        if hdul[0].header.get("SCANTYPE") != "FIBER_DM_PISTON":
            raise ValueError("Select a FITS from the new all-segment DM/focus acquisition script")
        meta = json.loads(hdul["SCAN_SETTINGS"].data["JSON"][0])
        data = {key.lower(): np.array(hdul[key].data, copy=True)
                for key in ("SPECTRA", "ACQUIRED", "FOCUS_CMD", "PISTON_CMD", "OPD", "SEGMENT", "SPECTRAL_X")}
        saved_flux = np.array(hdul["FLUX"].data, dtype=float)
    expected = (len(data["focus_cmd"]), 37, len(data["piston_cmd"]), len(data["spectral_x"]))
    if data["spectra"].shape != expected or data["acquired"].shape != expected[:-1]:
        raise ValueError(f"Unexpected cube / mask shape; expected {expected}")
    if not np.array_equal(data["segment"], np.arange(37)):
        raise ValueError("Expected segments 0..36 in order")
    if not np.allclose(data["opd"], 2*data["piston_cmd"]):
        raise ValueError("OPD is not twice commanded DM stroke")
    data["acquired"] = data["acquired"].astype(bool)
    data["flux"] = data["spectra"].sum(axis=-1)
    if not np.allclose(data["flux"][data["acquired"]], saved_flux[data["acquired"]]):
        raise ValueError("Stored FLUX and integrated spectra disagree")
    data["flux"][~data["acquired"]] = np.nan
    data["metadata"] = meta
    return data


def synthetic_demo():
    rng = np.random.default_rng(7)
    z = np.arange(1500, 2001, 100, dtype=float)
    p = np.linspace(-0.6, 0.3, 19)
    opd = 2*p
    v = 0.03 + 0.55*np.exp(-((z[:, None]-1750)/220)**2)*(0.3+0.7*np.arange(37)[None, :]/36)
    phase = np.arange(37)*0.15
    flux = 5000*(1+v[:, :, None]*np.cos(2*np.pi*opd[None, None, :]/1.6+phase[None, :, None]))
    flux += rng.normal(0, 2, flux.shape)
    profile = np.exp(-0.5*((np.arange(109)-54)/35)**2)
    spectra = flux[..., None]*profile/profile.sum()
    return dict(spectra=spectra, flux=flux, acquired=np.ones(flux.shape, dtype=bool),
                focus_cmd=z, piston_cmd=p, opd=opd, segment=np.arange(37),
                spectral_x=np.arange(158, 267), metadata={"status":"SYNTHETIC DEMO", "nframes":25})


if USE_SYNTHETIC_DEMO:
    scan = synthetic_demo()
    source_path = None
    source_label = "SYNTHETIC DEMO - not instrument measurements"
    output_dir = Path(OUTPUT_DIR) if OUTPUT_DIR is not None else Path.cwd()/"synthetic_dm_piston_analysis"
else:
    source_path = selected_scan_path()
    scan = load_scan(source_path)
    source_label = source_path.name
    output_dir = Path(OUTPUT_DIR) if OUTPUT_DIR is not None else source_path.parent/(source_path.stem+"_analysis")
output_dir.mkdir(parents=True, exist_ok=True)
focus = scan["focus_cmd"]
opd = scan["opd"]
flux = scan["flux"]
print(source_label)
print("Cube [focus,segment,piston,column]:", scan["spectra"].shape)
print("Completed:", scan["acquired"].sum(), "/", scan["acquired"].size, "status:", scan["metadata"]["status"])
print("Outputs:", output_dir.resolve())


## Fringe fitting
For each focus and segment, fit the integrated 109-column spectrum versus OPD using
`F = m + B*cos(2*k*x) + C*sin(2*k*x)`, equivalent to the reference sine-squared dip.
Then `c0=m+hypot(B,C)`, `A=2*hypot(B,C)` and `V=hypot(B,C)/m`.
A frequency grid avoids sensitivity to nonlinear initial guesses. Fit quality, residual
RMS and frequency-bound flags are exported. Rejected/insufficient fits retain NaN visibility;
flat positive traces have V=0. The contrast is model-based; it is not a noise-bias-corrected
visibility or a calibrated uncertainty estimate. Narrow OPD coverage can leave period and
contrast weakly constrained; inspect residuals and bound flags before interpreting a map.


In [ ]:
def fit_fringe(x, y):
    good = np.isfinite(x) & np.isfinite(y)
    x, y = np.asarray(x)[good], np.asarray(y)[good]
    result = dict(visibility=np.nan, visibility_raw=np.nan, mean_flux=float(y.mean()) if y.size else np.nan,
                  c0=np.nan, A=np.nan, k=np.nan, phi=np.nan, x_center=np.nan,
                  r2=np.nan, residual_rms=np.nan, period_um=np.nan, n_points=len(x),
                  fit_success=False, frequency_at_bound=False, fit_reason="too_few_points")
    if len(x) < MIN_POINTS_FOR_FIT or np.ptp(x) <= 0:
        return result
    if y.mean() <= 0:
        result["fit_reason"] = "nonpositive_mean_flux"
        return result
    if np.ptp(y)/abs(y.mean()) < FLAT_FRACTION_THRESHOLD:
        result.update(visibility=0., visibility_raw=0., c0=float(y.mean()), A=0.,
                      residual_rms=float(np.std(y)), fit_reason="flat_below_threshold")
        return result
    period_bounds = OPD_PERIOD_BOUNDS_UM or (np.ptp(x)/1.8, np.ptp(x)/0.6)
    lo, hi = map(float, period_bounds)
    if not 0 < lo < hi or not np.all(np.isfinite([lo, hi])):
        raise ValueError("OPD_PERIOD_BOUNDS_UM must be two finite positive increasing values")
    if N_FREQUENCY_TRIALS < 3:
        raise ValueError("N_FREQUENCY_TRIALS must be at least 3")
    x0 = x-x.mean()
    k_grid = np.linspace(np.pi/hi, np.pi/lo, N_FREQUENCY_TRIALS)
    best = None
    for index, k in enumerate(k_grid):
        X = np.column_stack([np.ones_like(x0), np.cos(2*k*x0), np.sin(2*k*x0)])
        beta, _, rank, _ = np.linalg.lstsq(X, y, rcond=None)
        if rank != 3:
            continue
        rss = float(np.sum((y-X@beta)**2))
        if best is None or rss < best[0]:
            best = (rss, index, k, beta)
    if best is None:
        result["fit_reason"] = "rank_deficient"
        return result
    rss, index, k, (mean, B, C) = best
    amplitude = float(np.hypot(B, C))
    visibility = float(amplitude/mean) if mean > 0 else np.nan
    r2 = 1-rss/np.sum((y-y.mean())**2)
    result.update(visibility_raw=visibility, c0=float(mean+amplitude), A=2*amplitude,
                  k=float(k), phi=float(-0.5*np.arctan2(C, B)), x_center=float(x.mean()),
                  r2=float(r2), residual_rms=float(np.sqrt(rss/len(y))), period_um=float(np.pi/k),
                  frequency_at_bound=index in (0, len(k_grid)-1))
    if not np.isfinite(visibility) or visibility < 0 or visibility > MAX_VISIBILITY or mean-amplitude < 0:
        result["fit_reason"] = "unphysical_contrast"
    elif r2 < MIN_FIT_R2:
        result["fit_reason"] = "low_r2"
    else:
        result.update(visibility=visibility, fit_success=True, fit_reason="accepted")
    return result


def evaluate_fit(result, x):
    if result["fit_reason"] == "flat_below_threshold":
        return np.full_like(x, result["mean_flux"], dtype=float)
    if not np.isfinite(result["k"]):
        return np.full_like(x, np.nan, dtype=float)
    return result["c0"]-result["A"]*np.sin(result["k"]*(x-result["x_center"])+result["phi"])**2


fit_results = {}
records = []
for iz, z in enumerate(focus):
    for segment in range(37):
        result = fit_fringe(opd, flux[iz, segment])
        fit_results[iz, segment] = result
        records.append(dict(focus_index=iz, focus_z_pulses=z, segment=segment, **result))
results = pd.DataFrame(records)
results.to_csv(output_dir/"visibility_fit_results.csv", index=False)
settings = {name: globals()[name] for name in (
    "OPD_PERIOD_BOUNDS_UM", "N_FREQUENCY_TRIALS", "MIN_POINTS_FOR_FIT", "FLAT_FRACTION_THRESHOLD",
    "MIN_FIT_R2", "MAX_VISIBILITY", "VISIBILITY_CMAP", "VISIBILITY_LIMITS", "VISIBILITY_LOG_SCALE")}
(output_dir/"analysis_settings.json").write_text(json.dumps(dict(source=source_label, settings=settings), indent=2), encoding="utf-8")
visibility = results["visibility"].to_numpy().reshape(len(focus), 37)
print(results.groupby("fit_reason").size().to_string())
display(results.head())


## One DM visibility map per focus
Numbering follows the reference: columns of 4, 5, 6, 7, 6, 5, 4 segments,
top-to-bottom within each column, proceeding right-to-left. Segment labels identify
the piston being swept; all maps use the single saved dispersed fiber output.


In [ ]:
def dm_geometry():
    radius = 1/np.sqrt(3)
    centers = []
    patches = []
    for column, count in enumerate([4, 5, 6, 7, 6, 5, 4]):
        for y in np.arange(count)[::-1]-(count-1)/2:
            center = np.array([(3-column)*1.5*radius, y])
            angles = np.arange(6)*np.pi/3
            vertices = center + radius*np.column_stack([np.cos(angles), np.sin(angles)])
            centers.append(center)
            patches.append(Polygon(vertices, closed=True))
    return np.array(centers), patches


def draw_dm_map(iz):
    centers, patches = dm_geometry()
    cmap = plt.get_cmap(VISIBILITY_CMAP).copy()
    cmap.set_bad("lightgrey")
    if VISIBILITY_LOG_SCALE:
        if VISIBILITY_LIMITS[0] <= 0:
            raise ValueError("Log visibility scale needs a positive lower limit")
        norm = LogNorm(*VISIBILITY_LIMITS)
        values = np.where(np.isfinite(visibility[iz]), np.maximum(visibility[iz], VISIBILITY_LIMITS[0]), np.nan)
    else:
        norm = Normalize(*VISIBILITY_LIMITS)
        values = visibility[iz]
    with plt.ioff():  # suppress backend auto-display before the explicit output
        fig, ax = plt.subplots(figsize=(7.2, 7.0))
    collection = PatchCollection(patches, cmap=cmap, norm=norm, edgecolor="black", linewidth=.8)
    collection.set_array(np.ma.masked_invalid(values))
    ax.add_collection(collection)
    for segment, (x, y) in enumerate(centers):
        ax.text(x, y, str(segment), ha="center", va="center", fontsize=9, color="white",
                bbox=dict(facecolor="black", alpha=.35, edgecolor="none", pad=.6))
    ax.set(xlim=(-3.6, 3.6), ylim=(-4, 4), aspect="equal")
    ax.set_axis_off()
    ax.set_title(f"Single-fiber fringe visibility - Z = {focus[iz]:g} pulses\n"
                 f"{np.isfinite(visibility[iz]).sum()}/37 valid; grey = missing/rejected", fontsize=12)
    fig.colorbar(collection, ax=ax, shrink=.8, label="Visibility (Fmax - Fmin) / (Fmax + Fmin)")
    fig.text(.5, .02, source_label, ha="center", fontsize=8)
    fig.subplots_adjust(bottom=.09, top=.88)
    return fig


for iz in range(len(focus)):
    fig = draw_dm_map(iz)
    if SAVE_FOCUS_PNGS:
        fig.savefig(output_dir/f"visibility_focus_{iz:03d}_z{focus[iz]:g}.png", dpi=160)
    display_analysis_figure(fig)


## Per-segment focus / OPD inspector
Use the dropdowns below, or call `inspect_segment(segment=12, focus_index=2)` directly.
The first panel is measured broadband flux versus **focus and OPD**; the second is the
selected fringe with the sine-squared fit and residuals; the third is visibility versus
focus. Optional row normalization affects only the first panel, never the fitted data.
Unacquired samples remain gaps. Rejected candidate fits are dashed and retain their
reason; CSV stores the raw candidate contrast separately from accepted visibility.


In [ ]:
def cell_edges(values, singleton_width=1.):
    values = np.asarray(values, dtype=float)
    if len(values) == 1:
        return np.array([values[0]-singleton_width/2, values[0]+singleton_width/2])
    mid = (values[:-1]+values[1:])/2
    return np.r_[values[0]-(mid[0]-values[0]), mid, values[-1]+(values[-1]-mid[-1])]


def plot_segment_diagnostics(segment=SELECTED_SEGMENT, focus_index=SELECTED_FOCUS_INDEX,
                             normalize_rows=NORMALIZE_FOCUS_OPD_ROWS):
    segment, focus_index = int(segment), int(focus_index)
    if not 0 <= segment < 37 or not 0 <= focus_index < len(focus):
        raise ValueError("Invalid segment ID or focus index")
    with plt.ioff():  # suppress backend auto-display before the explicit output
        fig = plt.figure(figsize=(16, 6.2), constrained_layout=True)
    grid = fig.add_gridspec(2, 3, height_ratios=(3, 1))
    ax_heat = fig.add_subplot(grid[:, 0])
    ax_trace = fig.add_subplot(grid[0, 1])
    ax_resid = fig.add_subplot(grid[1, 1], sharex=ax_trace)
    ax_v = fig.add_subplot(grid[:, 2])
    displayed = flux[:, segment].copy()
    if normalize_rows:
        count = np.isfinite(displayed).sum(axis=1, keepdims=True)
        means = np.divide(np.nansum(displayed, axis=1, keepdims=True), count,
                          out=np.full((len(focus), 1), np.nan), where=count > 0)
        displayed = np.divide(displayed, means, out=np.full_like(displayed, np.nan), where=means != 0)
    # Sort display coordinates only; result indices always follow acquisition order.
    iz_sort, ip_sort = np.argsort(focus), np.argsort(opd)
    image = ax_heat.pcolormesh(cell_edges(opd[ip_sort], .1), cell_edges(focus[iz_sort]),
                               displayed[np.ix_(iz_sort, ip_sort)], shading="flat", cmap="viridis")
    ax_heat.axhline(focus[focus_index], color="white", ls="--", lw=1)
    ax_heat.set(xlabel="OPD = 2 x DM stroke [um]", ylabel="Absolute mount Z [pulses]",
                title="Measured focus versus OPD")
    fig.colorbar(image, ax=ax_heat, label="Flux / mean at each focus" if normalize_rows else "ADU/frame")
    y = flux[focus_index, segment]
    result = fit_results[focus_index, segment]
    dense = np.linspace(opd.min(), opd.max(), 500)
    ax_trace.plot(opd, y, "o", ms=4, label="Measured")
    ax_trace.plot(dense, evaluate_fit(result, dense), color="tab:orange",
                  ls="-" if result["fit_success"] else "--", label="Fit / candidate")
    ax_trace.set(ylabel="Integrated flux [ADU/frame]", title=f"Z={focus[focus_index]:g}: {result['fit_reason']}")
    ax_trace.legend(fontsize=8)
    ax_trace.text(.02, .04, f"V={result['visibility']:.4g}; R^2={result['r2']:.3g}\n"
                  f"Period={result['period_um']:.3g} um; frequency at bound={result['frequency_at_bound']}",
                  transform=ax_trace.transAxes, fontsize=8)
    ax_resid.axhline(0, color="grey", lw=.8)
    ax_resid.plot(opd, y-evaluate_fit(result, opd), "o", ms=3)
    ax_resid.set(xlabel="OPD = 2 x DM stroke [um]", ylabel="Residual")
    ax_v.plot(focus[iz_sort], visibility[iz_sort, segment], "o-", color="tab:purple")
    ax_v.axvline(focus[focus_index], color="grey", ls="--")
    ax_v.set(xlabel="Absolute mount Z [pulses]", ylabel="Fitted visibility", ylim=(0, MAX_VISIBILITY*1.05),
             title="Visibility versus focus")
    ax_v.grid(alpha=.25)
    fig.suptitle(f"DM segment {segment} - {source_label}", fontsize=13)
    return fig


def inspect_segment(segment=SELECTED_SEGMENT, focus_index=SELECTED_FOCUS_INDEX,
                    normalize_rows=NORMALIZE_FOCUS_OPD_ROWS):
    fig = plot_segment_diagnostics(segment, focus_index, normalize_rows)
    display_analysis_figure(fig)


# Close the previous tool when this cell is rerun.
for _widget_name in ("segment_inspector_ui", "inspector"):
    _previous_widget = globals().get(_widget_name)
    if _previous_widget is not None:
        _previous_widget.close()

_segment_widgets_available = False
if USE_INTERACTIVE_WIDGETS:
    try:
        import ipywidgets as widgets
    except ImportError:
        print("ipywidgets unavailable; displaying one static inspector.")
    else:
        _segment_widgets_available = True

if _segment_widgets_available:
    controls = dict(segment=widgets.Dropdown(options=list(range(37)), value=SELECTED_SEGMENT, description="Segment"),
                    focus_index=widgets.Dropdown(options=[(f"{i}: Z={z:g}", i) for i, z in enumerate(focus)],
                                                 value=SELECTED_FOCUS_INDEX, description="Focus"),
                    normalize_rows=widgets.Checkbox(value=NORMALIZE_FOCUS_OPD_ROWS, description="Normalize map rows"))
    inspector = widgets.interactive_output(inspect_segment, controls)
    segment_inspector_ui = widgets.VBox([widgets.HBox(list(controls.values())), inspector])
    display(segment_inspector_ui)
else:
    inspect_segment()

if SAVE_SEGMENT_PNGS:
    for segment in range(37):
        fig = plot_segment_diagnostics(segment)
        fig.savefig(output_dir/f"segment_{segment:02d}_focus_opd.png", dpi=130)
        plt.close(fig)
    print("Saved all 37 per-segment focus/OPD diagnostics.")


## MP4 export
One numbered visibility DM map per focus, in acquisition order, with a common colour
scale. `imageio-ffmpeg` supplies an encoder if system FFmpeg is unavailable. Set
`EXPORT_MP4=False` to skip. The filename contains `synthetic` in preview mode.


In [ ]:
def export_visibility_movie(path):
    executable = shutil.which("ffmpeg")
    if executable is None:
        try:
            import imageio_ffmpeg
            executable = imageio_ffmpeg.get_ffmpeg_exe()
        except (ImportError, RuntimeError) as error:
            raise RuntimeError("MP4 export needs FFmpeg on PATH or the imageio-ffmpeg package") from error
    if MOVIE_FPS <= 0 or MOVIE_HOLD_FRAMES < 1 or int(MOVIE_HOLD_FRAMES) != MOVIE_HOLD_FRAMES:
        raise ValueError("Set positive MOVIE_FPS and positive integer MOVIE_HOLD_FRAMES")
    writer = FFMpegWriter(fps=MOVIE_FPS, codec="libx264", extra_args=["-pix_fmt", "yuv420p"],
                          metadata={"title": f"Fiber DM visibility - {source_label}"})
    # Reuse one figure/canvas for consistent dimensions across every focus frame.
    fig = draw_dm_map(0)
    try:
        with plt.rc_context({"animation.ffmpeg_path": executable}):
            with writer.saving(fig, str(path), dpi=120):
                ax = fig.axes[0]
                collection = ax.collections[0]
                for iz in range(len(focus)):
                    values = visibility[iz]
                    if VISIBILITY_LOG_SCALE:
                        values = np.where(np.isfinite(values), np.maximum(values, VISIBILITY_LIMITS[0]), np.nan)
                    collection.set_array(np.ma.masked_invalid(values))
                    ax.set_title(f"Single-fiber fringe visibility - Z = {focus[iz]:g} pulses\n"
                                 f"{np.isfinite(visibility[iz]).sum()}/37 valid; grey = missing/rejected", fontsize=12)
                    for _ in range(int(MOVIE_HOLD_FRAMES)):
                        writer.grab_frame(facecolor="white")
    finally:
        plt.close(fig)
    return path


if EXPORT_MP4:
    movie_path = output_dir/("synthetic_visibility_vs_focus.mp4" if USE_SYNTHETIC_DEMO else "visibility_vs_focus.mp4")
    export_visibility_movie(movie_path)
    print("Saved:", movie_path)
    display(Video(str(movie_path), embed=True))


## Spectra overlay: choose segment, focus and piston samples
At one DM segment and one absolute mount-Z focus, overlay the saved spectra with
**wavelength [nm] on X** and **flux [ADU/frame per detector column] on Y**.
Each piston has a different colour and is labelled with both DM stroke and OPD.
Default sampling is **plot one, skip two**: indices `0, 3, 6, 9, 12, 15, 18`,
so the usual 19-point scan gives seven curves, including both endpoints.
Change **Every nth** or the starting index, or pass an explicit list to
`show_piston_spectra(segment=12, focus_index=2, piston_indices=[0, 3, 6, 9, 12, 15, 18])`.

Flux is the saved dark-subtracted, frame-averaged row sum: no additional dark
subtraction, averaging, normalization or box-height factor is applied.
Wavelengths use the same embedded calibration snapshot as the focus/piston notebooks.
The mapping at the fiber row is inferred from the cross-channel trace model,
not a direct fiber calibration. Columns outside calibration support are omitted
without extrapolation. Missing piston acquisitions are skipped and reported.


In [ ]:
# SPECTRA OVERLAY CONTROLS - standalone settings for the final tool
SPECTRA_SEGMENT = 0
SPECTRA_FOCUS_INDEX = 0  # zero-based index; dropdown also shows absolute mount Z
SPECTRA_PISTON_STRIDE = 3  # plot one, skip two -> seven of the usual 19 samples
SPECTRA_PISTON_START_INDEX = 0
SPECTRA_PISTON_INDICES = None  # optional explicit indices; overrides stride/start
SPECTRA_CMAP = "turbo"
SPECTRA_SAVE_PNG = False  # True saves the displayed selection under output_dir
WAVELENGTH_CALIBRATION_PATH = None  # None: embedded focus/piston snapshot; or wavelength_solution.json


In [ ]:
_SPECTRA_CALIBRATION_SNAPSHOT = json.loads(r'''
{
  "source_file": "C:\\Users\\mcol0021\\OneDrive - The University of Sydney (Staff)\\Desktop\\GLINT_test reports\\GLINT_APAPANE_wavelength_calibration\\calibration_output\\wavelength_solution.json",
  "source_sha256": "380c29d43c923593801cf4c8acf75733c88a094359e28a27145222070b6f147e",
  "created_utc": "2026-09-08T05:22:42.893810+00:00",
  "schema_version": 2,
  "trace_models": [
    {
      "wavelength_nm": 1200.0,
      "coefficients_for_normalized_y_ascending_power": [
        110.72207128729045,
        -2.736150158588414,
        -5.787724401628924
      ],
      "rms_pixel": 0.22723745857337516,
      "support_channel_ids": [
        "phot_1",
        "phot_2",
        "phot_3",
        "n1_11_31_ft1",
        "n1_11_31_ft2",
        "n2_11_20_ft1",
        "n2_11_20_ft2",
        "n3_20_31_ft1",
        "n3_20_31_ft2"
      ]
    },
    {
      "wavelength_nm": 1250.0,
      "coefficients_for_normalized_y_ascending_power": [
        136.2605656129342,
        -2.8499297816329614,
        -5.496905001725147
      ],
      "rms_pixel": 0.17221116103738499,
      "support_channel_ids": [
        "phot_1",
        "phot_2",
        "phot_3",
        "n1_11_31_ft1",
        "n1_11_31_ft2",
        "n2_11_20_ft1",
        "n2_11_20_ft2",
        "n3_20_31_ft1",
        "n3_20_31_ft2"
      ]
    },
    {
      "wavelength_nm": 1300.0,
      "coefficients_for_normalized_y_ascending_power": [
        157.9734578998678,
        -2.750759195219552,
        -5.072523492556756
      ],
      "rms_pixel": 0.1403854066272288,
      "support_channel_ids": [
        "phot_1",
        "phot_2",
        "phot_3",
        "n1_11_31_ft1",
        "n1_11_31_ft2",
        "n2_11_20_ft1",
        "n2_11_20_ft2",
        "n3_20_31_ft1",
        "n3_20_31_ft2"
      ]
    },
    {
      "wavelength_nm": 1350.0,
      "coefficients_for_normalized_y_ascending_power": [
        177.3940680253761,
        -2.739949497747996,
        -5.408767669263737
      ],
      "rms_pixel": 0.05539905117961564,
      "support_channel_ids": [
        "phot_1",
        "phot_2",
        "phot_3",
        "n1_11_31_ft1",
        "n1_11_31_ft2",
        "n2_11_20_ft1",
        "n2_11_20_ft2",
        "n3_20_31_ft1",
        "n3_20_31_ft2"
      ]
    },
    {
      "wavelength_nm": 1400.0,
      "coefficients_for_normalized_y_ascending_power": [
        194.1550960411336,
        -2.8150469488888588,
        -5.048542797952065
      ],
      "rms_pixel": 0.11225106721543762,
      "support_channel_ids": [
        "phot_1",
        "phot_2",
        "phot_3",
        "n1_11_31_ft1",
        "n1_11_31_ft2",
        "n2_11_20_ft1",
        "n2_11_20_ft2",
        "n3_20_31_ft1",
        "n3_20_31_ft2"
      ]
    },
    {
      "wavelength_nm": 1450.0,
      "coefficients_for_normalized_y_ascending_power": [
        209.2811008440594,
        -2.7789519097817794,
        -5.235754140637014
      ],
      "rms_pixel": 0.09995393915354084,
      "support_channel_ids": [
        "phot_1",
        "phot_2",
        "phot_3",
        "n1_11_31_ft1",
        "n1_11_31_ft2",
        "n2_11_20_ft1",
        "n2_11_20_ft2",
        "n3_20_31_ft1",
        "n3_20_31_ft2"
      ]
    },
    {
      "wavelength_nm": 1500.0,
      "coefficients_for_normalized_y_ascending_power": [
        222.65152419154893,
        -2.676626449540458,
        -5.113894101749004
      ],
      "rms_pixel": 0.04082303189581721,
      "support_channel_ids": [
        "phot_1",
        "phot_2",
        "phot_3",
        "n1_11_31_ft1",
        "n1_11_31_ft2",
        "n2_11_20_ft1",
        "n2_11_20_ft2",
        "n3_20_31_ft1",
        "n3_20_31_ft2"
      ]
    },
    {
      "wavelength_nm": 1550.0,
      "coefficients_for_normalized_y_ascending_power": [
        234.5747956825917,
        -2.664031942387192,
        -5.04497819535988
      ],
      "rms_pixel": 0.02676477131138072,
      "support_channel_ids": [
        "phot_1",
        "phot_2",
        "phot_3",
        "n1_11_31_ft1",
        "n1_11_31_ft2",
        "n2_11_20_ft1",
        "n2_11_20_ft2",
        "n3_20_31_ft1",
        "n3_20_31_ft2"
      ]
    },
    {
      "wavelength_nm": 1600.0,
      "coefficients_for_normalized_y_ascending_power": [
        245.48366490406644,
        -2.6219724556608943,
        -5.090748483992172
      ],
      "rms_pixel": 0.11822341147229093,
      "support_channel_ids": [
        "phot_1",
        "phot_2",
        "phot_3",
        "n1_11_31_ft1",
        "n1_11_31_ft2",
        "n2_11_20_ft1",
        "n2_11_20_ft2",
        "n3_20_31_ft1",
        "n3_20_31_ft2"
      ]
    },
    {
      "wavelength_nm": 1650.0,
      "coefficients_for_normalized_y_ascending_power": [
        255.0674543477981,
        -2.702844921992064,
        -5.131617958101788
      ],
      "rms_pixel": 0.07277331150191564,
      "support_channel_ids": [
        "phot_1",
        "phot_2",
        "phot_3",
        "n1_11_31_ft1",
        "n1_11_31_ft2",
        "n2_11_20_ft1",
        "n2_11_20_ft2",
        "n3_20_31_ft1",
        "n3_20_31_ft2"
      ]
    },
    {
      "wavelength_nm": 1700.0,
      "coefficients_for_normalized_y_ascending_power": [
        266.3825872729101,
        -2.6933287543329154,
        -5.024773570284092
      ],
      "rms_pixel": 0.039178177860414276,
      "support_channel_ids": [
        "phot_1",
        "phot_2",
        "phot_3",
        "n1_11_31_ft1",
        "n1_11_31_ft2",
        "n2_11_20_ft1",
        "n2_11_20_ft2",
        "n3_20_31_ft1",
        "n3_20_31_ft2"
      ]
    }
  ],
  "channels": {
    "phot_1": {
      "y_center_pixel": 242.5
    },
    "phot_2": {
      "y_center_pixel": 224.5
    },
    "phot_3": {
      "y_center_pixel": 205.5
    },
    "n1_11_31_ft1": {
      "y_center_pixel": 186.5
    },
    "n1_11_31": {
      "y_center_pixel": 168.5
    },
    "n1_11_31_ft2": {
      "y_center_pixel": 149.5
    },
    "n2_11_20_ft1": {
      "y_center_pixel": 130.5
    },
    "n2_11_20": {
      "y_center_pixel": 112.0
    },
    "n2_11_20_ft2": {
      "y_center_pixel": 93.5
    },
    "n3_20_31_ft1": {
      "y_center_pixel": 55.5
    },
    "n3_20_31": {
      "y_center_pixel": 37.5
    },
    "n3_20_31_ft2": {
      "y_center_pixel": 18.5
    }
  },
  "coordinate_conventions": {
    "detector_shape_yx": [
      256,
      320
    ],
    "x_pixel": "zero-indexed detector-column centre",
    "y_slices": "Python half-open [y0:y1]; x limits in detector_boxes.json are intentionally ignored"
  }
}
''')


In [ ]:
def spectra_wavelength_axis(data, calibration_path=None):
    if calibration_path is None:
        solution = _SPECTRA_CALIBRATION_SNAPSHOT
        calibration_label = "embedded focus/piston calibration snapshot"
    else:
        path = Path(calibration_path).expanduser().resolve()
        solution = json.loads(path.read_text(encoding="utf-8"))
        calibration_label = str(path)
    if solution.get("schema_version") != 2 or not solution.get("trace_models"):
        raise ValueError("Expected schema-2 wavelength_solution.json with trace_models")
    box = data["metadata"].get("spectral_box_y0_y1_x0_x1")
    if box is None:
        if not USE_SYNTHETIC_DEMO:
            raise ValueError("Missing spectral extraction box in saved scan metadata")
        box = (72, 77, 158, 267)  # the explicitly labelled synthetic demo detector box
    y0, y1, _, _ = box
    detector_y = (float(y0)+float(y1)-1)/2  # mean of the included pixel-row centres
    channels = solution["channels"]
    centres = np.array([channel["y_center_pixel"] for channel in channels.values()], dtype=float)
    traces = sorted(solution["trace_models"], key=lambda trace: trace["wavelength_nm"])
    support_ids = {key for trace in traces for key in trace["support_channel_ids"]}
    support_y = np.array([channels[key]["y_center_pixel"] for key in support_ids], dtype=float)
    if not support_y.size or not support_y.min() <= detector_y <= support_y.max():
        raise ValueError(f"Fiber row Y={detector_y:g} is outside the calibration trace support")
    yn = (detector_y-centres.mean()) / max(np.ptp(centres)/2, 1.)
    wavelengths = np.array([trace["wavelength_nm"] for trace in traces], dtype=float)
    pixels = np.array([np.polynomial.polynomial.polyval(
        yn, trace["coefficients_for_normalized_y_ascending_power"]) for trace in traces])
    if (not np.all(np.isfinite(pixels)) or not np.all(np.isfinite(wavelengths))
            or not np.all(np.diff(pixels) > 0) or not np.all(np.diff(wavelengths) > 0)):
        raise ValueError("Expected finite increasing wavelength/pixel calibration knots")
    wavelength_nm = np.interp(np.asarray(data["spectral_x"], dtype=float), pixels, wavelengths,
                              left=np.nan, right=np.nan)
    if np.isfinite(wavelength_nm).sum() < 2:
        raise ValueError("Fewer than two recorded columns fall within wavelength calibration support")
    return wavelength_nm, detector_y, calibration_label


def spectra_piston_indices(count, stride=3, start_index=0, piston_indices=None):
    if piston_indices is not None:
        indices = np.asarray(piston_indices, dtype=float)
        if (indices.ndim != 1 or not indices.size or not np.all(np.isfinite(indices))
                or not np.all(indices == np.rint(indices))):
            raise ValueError("piston_indices must be a nonempty list of integer sample indices")
        indices = indices.astype(int)
        if len(np.unique(indices)) != len(indices):
            raise ValueError("piston_indices must not contain duplicates")
    else:
        if not np.isfinite(stride) or int(stride) != stride or stride < 1:
            raise ValueError("Piston stride must be a positive integer")
        if not np.isfinite(start_index) or int(start_index) != start_index:
            raise ValueError("Starting sample index must be an integer")
        if not 0 <= start_index < count:
            raise ValueError(f"Starting sample index must be in 0..{count-1}")
        indices = np.arange(int(start_index), count, int(stride))
    if np.any(indices < 0) or np.any(indices >= count):
        raise ValueError(f"Piston sample indices must be in 0..{count-1}")
    return indices


spectra_wavelength_nm, spectra_detector_y, spectra_calibration_label = spectra_wavelength_axis(
    scan, WAVELENGTH_CALIBRATION_PATH)
print("Wavelength axis:", spectra_calibration_label)
print(f"Inferred at fiber row Y={spectra_detector_y:g}; "
      f"{np.isfinite(spectra_wavelength_nm).sum()}/{len(spectra_wavelength_nm)} calibrated columns; "
      "no extrapolation")


def plot_piston_spectra(segment=SPECTRA_SEGMENT, focus_index=SPECTRA_FOCUS_INDEX,
                       stride=SPECTRA_PISTON_STRIDE, start_index=SPECTRA_PISTON_START_INDEX,
                       piston_indices=None, save_png=False):
    if int(segment) != segment or not 0 <= segment < 37:
        raise ValueError("Segment must be an integer ID in 0..36")
    if int(focus_index) != focus_index or not 0 <= focus_index < len(focus):
        raise ValueError(f"Focus index must be in 0..{len(focus)-1}")
    segment, focus_index = int(segment), int(focus_index)
    chosen = spectra_piston_indices(len(scan["piston_cmd"]), stride, start_index, piston_indices)
    spectral_cube = scan["spectra"]
    supported = np.isfinite(spectra_wavelength_nm)
    valid, skipped = [], []
    for ip in chosen:
        if (scan["acquired"][focus_index, segment, ip]
                and np.any(np.isfinite(spectral_cube[focus_index, segment, ip, supported]))):
            valid.append(int(ip))
        else:
            skipped.append(int(ip))
    with plt.ioff():  # suppress backend auto-display before the explicit output
        fig, ax = plt.subplots(figsize=(11.5, 6.0))
    for colour_index, ip in enumerate(valid):
        piston = scan["piston_cmd"][ip]
        colour = plt.get_cmap(SPECTRA_CMAP)(.05+.90*colour_index/max(len(valid)-1, 1))
        ax.plot(spectra_wavelength_nm[supported], spectral_cube[focus_index, segment, ip, supported],
                color=colour, lw=1.8,
                label=f"#{ip}: piston={piston:+.3f} um; OPD={2*piston:+.3f} um")
    ax.set(xlabel="Wavelength [nm]", ylabel="Flux [ADU/frame per detector column]",
           title=f"DM segment {segment} | Z={focus[focus_index]:g} pulses | {len(valid)} piston spectra")
    ax.grid(alpha=.22)
    if valid:
        ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8, frameon=False,
                  title="Sample index / commanded stroke")
    else:
        ax.text(.5, .5, "No acquired spectra for these piston samples", ha="center", va="center",
                transform=ax.transAxes)
    fig.text(.08, .055, f"{source_label}\nWavelength inferred at detector Y={spectra_detector_y:g}; no extrapolation.",
             fontsize=8, va="bottom")
    fig.subplots_adjust(left=.09, right=.68, bottom=.19, top=.87)
    details = dict(segment=segment, focus_index=focus_index, focus_z_pulses=float(focus[focus_index]),
                   requested_indices=chosen.tolist(), plotted_indices=valid, skipped_indices=skipped,
                   calibrated_columns=int(supported.sum()), saved_png=None)
    if save_png:
        suffix = "_".join(str(ip) for ip in chosen)
        destination = output_dir/f"spectra_segment_{segment:02d}_focus_{focus_index:03d}_samples_{suffix}.png"
        fig.savefig(destination, dpi=160, facecolor="white")
        details["saved_png"] = str(destination)
    return fig, details


def show_piston_spectra(segment=SPECTRA_SEGMENT, focus_index=SPECTRA_FOCUS_INDEX,
                       stride=SPECTRA_PISTON_STRIDE, start_index=SPECTRA_PISTON_START_INDEX,
                       piston_indices=None, save_png=False):
    fig, details = plot_piston_spectra(segment, focus_index, stride, start_index, piston_indices, save_png)
    display_analysis_figure(fig)
    print("Plotted sample indices:", details["plotted_indices"])
    if details["skipped_indices"]:
        print("Skipped unacquired/invalid sample indices:", details["skipped_indices"])
    if details["saved_png"]:
        print("Saved:", details["saved_png"])
    return details


# Widget initialization already draws the first selection; do not add a second preview.
for _widget_name in ("spectra_overlay_ui", "spectra_output"):
    _previous_widget = globals().get(_widget_name)
    if _previous_widget is not None:
        _previous_widget.close()

_spectra_widgets_available = False
if USE_INTERACTIVE_WIDGETS:
    try:
        import ipywidgets as spectra_widgets
    except ImportError:
        print("ipywidgets unavailable; displaying one static spectra overlay.")
    else:
        _spectra_widgets_available = True

if _spectra_widgets_available:
    spectra_controls = dict(
        segment=spectra_widgets.Dropdown(options=list(range(37)), value=SPECTRA_SEGMENT, description="Segment"),
        focus_index=spectra_widgets.Dropdown(options=[(f"{i}: Z={z:g}", i) for i,z in enumerate(focus)],
                                              value=SPECTRA_FOCUS_INDEX, description="Focus"),
        stride=spectra_widgets.BoundedIntText(value=SPECTRA_PISTON_STRIDE, min=1,
                                               max=max(len(scan["piston_cmd"]), SPECTRA_PISTON_STRIDE),
                                               description="Every nth"),
        start_index=spectra_widgets.BoundedIntText(value=SPECTRA_PISTON_START_INDEX, min=0,
                                                   max=len(scan["piston_cmd"])-1, description="Start index"),
        save_png=spectra_widgets.Checkbox(value=SPECTRA_SAVE_PNG, description="Save PNG"))
    def _update_spectra_overlay(**selection):
        global spectra_overlay_selection
        spectra_overlay_selection = show_piston_spectra(**selection, piston_indices=SPECTRA_PISTON_INDICES)
    spectra_output = spectra_widgets.interactive_output(_update_spectra_overlay, spectra_controls)
    spectra_overlay_ui = spectra_widgets.VBox([
        spectra_widgets.HTML("<b>Spectra overlay: select segment / focus, plot one every N piston samples</b>"),
        spectra_widgets.HBox([spectra_controls["segment"], spectra_controls["focus_index"]]),
        spectra_widgets.HBox([spectra_controls["stride"], spectra_controls["start_index"], spectra_controls["save_png"]]),
        spectra_output])
    display(spectra_overlay_ui)

else:
    spectra_overlay_selection = show_piston_spectra(
        SPECTRA_SEGMENT, SPECTRA_FOCUS_INDEX, SPECTRA_PISTON_STRIDE, SPECTRA_PISTON_START_INDEX,
        piston_indices=SPECTRA_PISTON_INDICES, save_png=SPECTRA_SAVE_PNG)


## Spectral ratios: uniform brightness changes or wavelength-dependent changes?
For a chosen segment and focus, form the **mean spectrum at each wavelength** over
all acquired piston samples, then plot `R_p(lambda) = S_p(lambda) / mean_p(S_p(lambda))`.
The default seven displayed curves still use **plot one, skip two**. By default the
reference mean uses **all acquired pistons**, so thinning the display does not change
the denominator. The **Mean over** control can instead use only the selected samples.
The mean always concerns one segment and one focus, not different segments or focuses.

- **Top panel:** spectrum / mean spectrum. A flat curve, even above or below 1,
  indicates a wavelength-independent brightness change; a slope or structure indicates
  that the spectral shape varies with piston.
- **Bottom panel:** remove each spectrum's constant brightness factor and plot
  `100 * (R_p / a_p - 1)`. Here `a_p = sum(mean * spectrum) / sum(mean**2)` is the
  least-squares scale factor over valid displayed columns. Uniform scaling gives zero.
  The table reports this scale factor and the residual shape RMS and peak-to-peak in percent.

Unacquired spectra never enter the mean. Missing/nonfinite samples remain gaps.
Nonpositive mean values and columns below the configurable mean-flux threshold are
masked; the default floor is 1% of the peak positive mean. No extrapolation or additional
dark subtraction is applied. Optional **Smooth ratios** and **Smooth shape** controls
apply a centred moving average to either displayed panel independently. **Window [cols]**
sets an odd number of spectral columns (default 7); this is not a fixed width in nm.
Each contiguous finite stretch is smoothed separately, using a shorter window at its
edges; missing/masked columns stay gaps and are never bridged. Smoothing is off by default.

Smoothing affects display only: the reference mean, fitted brightness factors, shape
metrics and original ratio CSV columns always use unsmoothed data. Exports also include
separately labelled display curves and the smoothing settings. The factor `a_p` is fitted
by minimizing `sum((S_p - a_p * mean)**2)` over retained finite spectral columns, with no
additive offset. Equivalently it is a mean of `S_p/mean` weighted by `mean**2`, rather than
an unweighted average of the ratio. Dividing by `a_p` removes that one constant scaling;
for `S_p=1.2*mean`, the raw ratio is 1.2 and the brightness-removed residual is zero.
 These are descriptive comparisons, not noise-corrected
shape estimates or statistical significance tests; the shared mean includes each spectrum.


In [ ]:
# SPECTRAL RATIO CONTROLS - independent of the raw-spectra overlay
RATIO_SEGMENT = SPECTRA_SEGMENT
RATIO_FOCUS_INDEX = SPECTRA_FOCUS_INDEX
RATIO_PISTON_STRIDE = 3
RATIO_PISTON_START_INDEX = 0
RATIO_PISTON_INDICES = None  # optional explicit list; overrides stride/start
RATIO_MEAN_SCOPE = "all"  # "all" acquired pistons, or "selected" displayed samples
RATIO_MIN_MEAN_FRACTION = 0.01  # mask mean <= 1% of peak positive mean; 0 retains all positive means
RATIO_MIN_MEAN_ADU = 0.0        # additional absolute mean-flux floor [ADU/frame/column]
RATIO_SAVE_EXPORTS = False      # True: PNG + wavelength-resolved ratios CSV + shape-summary CSV

# Optional display-only smoothing; factors and metrics remain unsmoothed.
RATIO_SMOOTH_RAW = False
RATIO_SMOOTH_SHAPE = False
RATIO_SMOOTH_WINDOW = 7  # positive odd number of detector columns; ignored when both smoothing toggles are off


In [ ]:
def compute_spectral_ratios(data, segment, focus_index, indices, wavelength_nm,
                            mean_scope="all", min_mean_fraction=0.01, min_mean_adu=0.0):
    if int(segment) != segment or not 0 <= segment < 37:
        raise ValueError("Segment must be an integer ID in 0..36")
    if int(focus_index) != focus_index or not 0 <= focus_index < len(data["focus_cmd"]):
        raise ValueError("Invalid focus index")
    if mean_scope not in ("all", "selected"):
        raise ValueError("mean_scope must be 'all' or 'selected'")
    if (not np.all(np.isfinite([min_mean_fraction, min_mean_adu]))
            or not 0 <= min_mean_fraction < 1 or min_mean_adu < 0):
        raise ValueError("Mean fraction must be in [0,1); absolute mean floor must be nonnegative")
    segment, focus_index = int(segment), int(focus_index)
    spectra = np.asarray(data["spectra"][focus_index, segment], dtype=float)
    wavelength_nm = np.asarray(wavelength_nm, dtype=float)
    if wavelength_nm.shape != (spectra.shape[1],):
        raise ValueError("Wavelength axis must match the saved spectral columns")
    supported = np.isfinite(wavelength_nm)
    chosen = spectra_piston_indices(len(spectra), piston_indices=indices)
    acquired = np.asarray(data["acquired"][focus_index, segment], dtype=bool)
    valid_sample = acquired & np.any(np.isfinite(spectra[:, supported]), axis=1)
    reference_indices = np.flatnonzero(valid_sample) if mean_scope == "all" else chosen[valid_sample[chosen]]
    reference = spectra[reference_indices]
    finite_reference = np.isfinite(reference)
    reference_count = finite_reference.sum(axis=0)
    mean = np.divide(np.where(finite_reference, reference, 0.).sum(axis=0), reference_count,
                     out=np.full(spectra.shape[1], np.nan), where=reference_count > 0)
    positive_mean = mean[supported & np.isfinite(mean) & (mean > 0)]
    peak = float(positive_mean.max()) if positive_mean.size else 0.
    floor = max(float(min_mean_adu), float(min_mean_fraction)*peak)
    valid_column = supported & np.isfinite(mean) & (mean > floor)
    plotted = chosen[valid_sample[chosen]]
    skipped = chosen[~valid_sample[chosen]]
    selected = spectra[plotted]
    good = valid_column[None, :] & np.isfinite(selected)
    ratios = np.divide(selected, mean[None, :], out=np.full(selected.shape, np.nan), where=good)
    shape_percent = np.full_like(ratios, np.nan)
    records = []
    for row, ip in enumerate(plotted):
        mask = good[row]
        scale = float(np.dot(mean[mask], selected[row, mask])/np.dot(mean[mask], mean[mask])) if mask.any() else np.nan
        if np.isfinite(scale) and scale > 0:
            shape_percent[row] = 100*(ratios[row]/scale-1)
        residual = shape_percent[row, np.isfinite(shape_percent[row])]
        records.append(dict(piston_index=int(ip), piston_um=float(data["piston_cmd"][ip]),
                            opd_um=float(2*data["piston_cmd"][ip]), scale_to_mean=scale,
                            shape_rms_percent=float(np.sqrt(np.mean(residual**2))) if residual.size else np.nan,
                            shape_peak_to_peak_percent=float(np.ptp(residual)) if residual.size else np.nan,
                            n_ratio_columns=int(mask.sum())))
    summary = pd.DataFrame(records, columns=["piston_index", "piston_um", "opd_um", "scale_to_mean",
                                              "shape_rms_percent", "shape_peak_to_peak_percent", "n_ratio_columns"])
    return dict(wavelength_nm=wavelength_nm, mean_spectrum=mean, reference_count=reference_count,
                valid_column=valid_column, ratios=ratios, shape_percent=shape_percent, summary=summary,
                requested_indices=chosen.tolist(), plotted_indices=plotted.tolist(), skipped_indices=skipped.tolist(),
                reference_indices=reference_indices.tolist(), mean_scope=mean_scope, floor_adu=floor,
                segment=segment, focus_index=focus_index)


def smooth_ratio_display(values, enabled=False, window=7):
    """Centred moving average within finite runs; never fills or bridges gaps."""
    values = np.asarray(values, dtype=float)
    if not enabled:
        return values.copy()  # disabled smoothing does not use or validate window
    if not np.isfinite(window) or window < 1 or int(window) != window or int(window) % 2 != 1:
        raise ValueError("Smoothing window must be a positive odd number of spectral columns")
    if values.ndim not in (1, 2):
        raise ValueError("Display smoothing expects one curve or a 2D collection of curves")
    result = values.copy()
    rows = result[None, :] if result.ndim == 1 else result
    half_width = int(window)//2
    for row in rows:
        finite = np.isfinite(row)
        starts = np.flatnonzero(np.diff(np.r_[False, finite].astype(int)) == 1)
        stops = np.flatnonzero(np.diff(np.r_[finite, False].astype(int)) == -1)+1
        for start, stop in zip(starts, stops):
            run = row[start:stop].copy()
            positions = np.arange(len(run))
            left = np.maximum(0, positions-half_width)
            right = np.minimum(len(run), positions+half_width+1)
            sums = np.r_[0., np.cumsum(run)]
            row[start:stop] = (sums[right]-sums[left])/(right-left)
    return result


def plot_piston_spectral_ratios(segment=RATIO_SEGMENT, focus_index=RATIO_FOCUS_INDEX,
                               stride=RATIO_PISTON_STRIDE, start_index=RATIO_PISTON_START_INDEX,
                               piston_indices=None, mean_scope=RATIO_MEAN_SCOPE,
                               min_mean_fraction=RATIO_MIN_MEAN_FRACTION, min_mean_adu=RATIO_MIN_MEAN_ADU,
                               save_exports=False, smooth_raw=RATIO_SMOOTH_RAW,
                               smooth_shape=RATIO_SMOOTH_SHAPE, smooth_window=RATIO_SMOOTH_WINDOW):
    indices = spectra_piston_indices(len(scan["piston_cmd"]), stride, start_index, piston_indices)
    result = compute_spectral_ratios(scan, segment, focus_index, indices, spectra_wavelength_nm,
                                     mean_scope, min_mean_fraction, min_mean_adu)
    result["display_ratios"] = smooth_ratio_display(result["ratios"], smooth_raw, smooth_window)
    result["display_shape_percent"] = smooth_ratio_display(result["shape_percent"], smooth_shape, smooth_window)
    result["display_smoothing"] = dict(raw=bool(smooth_raw), shape=bool(smooth_shape),
                                        window_columns=int(smooth_window) if smooth_raw or smooth_shape else None,
                                        method="centred moving average within finite runs",
                                        metrics="unsmoothed")
    smoothing_text = (f"Display smoothing: raw={'on' if smooth_raw else 'off'}, "
                      f"shape={'on' if smooth_shape else 'off'}")
    if smooth_raw or smooth_shape:
        smoothing_text += f"; {int(smooth_window)}-column moving average; metrics unsmoothed"
    with plt.ioff():  # suppress backend auto-display before the explicit output
        fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    for row, ip in enumerate(result["plotted_indices"]):
        colour = plt.get_cmap(SPECTRA_CMAP)(.05+.90*row/max(len(result["plotted_indices"])-1, 1))
        label = f"#{ip}: piston={scan['piston_cmd'][ip]:+.3f} um"
        axes[0].plot(spectra_wavelength_nm, result["display_ratios"][row], color=colour, lw=1.7, label=label)
        axes[1].plot(spectra_wavelength_nm, result["display_shape_percent"][row], color=colour, lw=1.7)
    axes[0].axhline(1, color="grey", ls="--", lw=1, label="Mean spectrum")
    axes[1].axhline(0, color="grey", ls="--", lw=1)
    axes[0].set(ylabel="Spectrum / mean spectrum", title="Raw ratios: flat curves indicate uniform brightness changes" + (" (smoothed display)" if smooth_raw else ""))
    axes[1].set(xlabel="Wavelength [nm]", ylabel="Shape residual after scaling [%]",
                title="Brightness factor removed: zero indicates unchanged spectral shape" + (" (smoothed display)" if smooth_shape else ""))
    for ax in axes:
        ax.grid(alpha=.22)
    axes[0].legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8, frameon=False)
    if not result["plotted_indices"] or not result["valid_column"].any():
        message = "No acquired spectra selected" if not result["plotted_indices"] else "All mean-spectrum columns are masked"
        axes[0].text(.5, .5, message, transform=axes[0].transAxes, ha="center")
    nreference = len(result["reference_indices"])
    fig.suptitle(f"DM segment {segment} | Z={scan['focus_cmd'][int(focus_index)]:g} pulses | "
                 f"mean of {nreference} piston samples ({mean_scope})", fontsize=12)
    fig.text(.09, .025, f"{source_label}\nInferred wavelength at Y={spectra_detector_y:g}; "
             f"{result['valid_column'].sum()}/{len(spectra_wavelength_nm)} columns retained; "
             f"mean-flux floor={result['floor_adu']:.4g} ADU/frame/column.\n{smoothing_text}", fontsize=8)
    fig.subplots_adjust(left=.10, right=.76, bottom=.13, top=.88, hspace=.28)
    result["saved_files"] = []
    if save_exports:
        sample_label = "_".join(str(ip) for ip in indices)
        stem = output_dir/f"ratios_segment_{int(segment):02d}_focus_{int(focus_index):03d}_{mean_scope}_samples_{sample_label}"
        if smooth_raw or smooth_shape:
            stem = stem.with_name(stem.name + f"_smooth_raw{int(bool(smooth_raw))}_shape{int(bool(smooth_shape))}_w{int(smooth_window)}")
        png_path = stem.with_suffix(".png")
        fig.savefig(png_path, dpi=160, facecolor="white")
        result["summary"].to_csv(str(stem)+"_summary.csv", index=False)
        table = pd.DataFrame(dict(wavelength_nm=spectra_wavelength_nm,
                                   mean_spectrum_adu_frame=result["mean_spectrum"],
                                   n_reference_samples=result["reference_count"],
                                   retained_column=result["valid_column"],
                                   raw_display_smoothed=bool(smooth_raw), shape_display_smoothed=bool(smooth_shape),
                                   smoothing_window_columns=result["display_smoothing"]["window_columns"]))
        for row, ip in enumerate(result["plotted_indices"]):
            table[f"ratio_piston_index_{ip}"] = result["ratios"][row]
            table[f"shape_percent_piston_index_{ip}"] = result["shape_percent"][row]
            table[f"display_ratio_piston_index_{ip}"] = result["display_ratios"][row]
            table[f"display_shape_percent_piston_index_{ip}"] = result["display_shape_percent"][row]
        table.to_csv(str(stem)+"_ratios.csv", index=False)
        result["saved_files"] = [str(png_path), str(stem)+"_summary.csv", str(stem)+"_ratios.csv"]
    return fig, result


def show_piston_spectral_ratios(**selection):
    fig, result = plot_piston_spectral_ratios(**selection)
    display_analysis_figure(fig)
    print("Displayed piston indices:", result["plotted_indices"], "| mean reference indices:", result["reference_indices"])
    if result["skipped_indices"]:
        print("Skipped unacquired/invalid indices:", result["skipped_indices"])
    print("Display smoothing:", result["display_smoothing"])
    display(result["summary"])
    for path in result["saved_files"]:
        print("Saved:", path)
    return result


# Widget callback OR static rendering: each tool draws once per selection.
for _widget_name in ("spectral_ratio_ui", "ratio_output"):
    _previous_widget = globals().get(_widget_name)
    if _previous_widget is not None:
        _previous_widget.close()

_ratio_widgets_available = False
if USE_INTERACTIVE_WIDGETS:
    try:
        import ipywidgets as ratio_widgets
    except ImportError:
        print("ipywidgets unavailable; displaying one static ratio plot.")
    else:
        _ratio_widgets_available = True

if _ratio_widgets_available:
    ratio_controls = dict(
        segment=ratio_widgets.Dropdown(options=list(range(37)), value=RATIO_SEGMENT, description="Segment"),
        focus_index=ratio_widgets.Dropdown(options=[(f"{i}: Z={z:g}", i) for i,z in enumerate(focus)],
                                            value=RATIO_FOCUS_INDEX, description="Focus"),
        stride=ratio_widgets.BoundedIntText(value=RATIO_PISTON_STRIDE, min=1,
                                             max=max(len(scan["piston_cmd"]), RATIO_PISTON_STRIDE), description="Every nth"),
        start_index=ratio_widgets.BoundedIntText(value=RATIO_PISTON_START_INDEX, min=0,
                                                 max=len(scan["piston_cmd"])-1, description="Start index"),
        mean_scope=ratio_widgets.Dropdown(options=[("All acquired pistons", "all"), ("Selected samples", "selected")],
                                           value=RATIO_MEAN_SCOPE, description="Mean over"),
        min_mean_fraction=ratio_widgets.BoundedFloatText(value=RATIO_MIN_MEAN_FRACTION, min=0., max=.99,
                                                          step=.01, description="Mean floor"),
        save_exports=ratio_widgets.Checkbox(value=RATIO_SAVE_EXPORTS, description="Save PNG/CSV"),
        smooth_raw=ratio_widgets.Checkbox(value=RATIO_SMOOTH_RAW, description="Smooth ratios"),
        smooth_shape=ratio_widgets.Checkbox(value=RATIO_SMOOTH_SHAPE, description="Smooth shape"),
        smooth_window=ratio_widgets.IntSlider(value=RATIO_SMOOTH_WINDOW, min=1,
                                               max=max(31, RATIO_SMOOTH_WINDOW), step=2,
                                               description="Window [cols]", continuous_update=False))
    def _update_spectral_ratios(**selection):
        global spectral_ratio_result
        spectral_ratio_result = show_piston_spectral_ratios(**selection, piston_indices=RATIO_PISTON_INDICES,
                                    min_mean_adu=RATIO_MIN_MEAN_ADU)
    ratio_output = ratio_widgets.interactive_output(_update_spectral_ratios, ratio_controls)
    spectral_ratio_ui = ratio_widgets.VBox([
        ratio_widgets.HTML("<b>Spectral ratios: brightness changes versus spectral shape</b>"),
        ratio_widgets.HBox([ratio_controls["segment"], ratio_controls["focus_index"], ratio_controls["mean_scope"]]),
        ratio_widgets.HBox([ratio_controls["stride"], ratio_controls["start_index"], ratio_controls["min_mean_fraction"],
                            ratio_controls["save_exports"]]),
        ratio_widgets.HBox([ratio_controls["smooth_raw"], ratio_controls["smooth_shape"], ratio_controls["smooth_window"]]),
        ratio_output])
    display(spectral_ratio_ui)

else:
    spectral_ratio_result = show_piston_spectral_ratios(
        segment=RATIO_SEGMENT, focus_index=RATIO_FOCUS_INDEX, stride=RATIO_PISTON_STRIDE,
        start_index=RATIO_PISTON_START_INDEX, piston_indices=RATIO_PISTON_INDICES,
        mean_scope=RATIO_MEAN_SCOPE, min_mean_fraction=RATIO_MIN_MEAN_FRACTION,
        min_mean_adu=RATIO_MIN_MEAN_ADU, save_exports=RATIO_SAVE_EXPORTS,
        smooth_raw=RATIO_SMOOTH_RAW, smooth_shape=RATIO_SMOOTH_SHAPE, smooth_window=RATIO_SMOOTH_WINDOW)


## Dispersed spectral maps across piston, at a fixed focus
Select **one segment and one absolute mount-Z focus**. Both maps have **wavelength
on X** (one saved detector column per spectral bin) and **DM surface piston [um] on Y**.
Colour shows:

- **Raw:** `S_p(lambda) / M(lambda)`, where `M` is the wavelength-wise mean over
  all acquired **piston samples** at this same segment and focus.
- **Brightness corrected:** `S_p(lambda) / (a_p * M(lambda))`, with one constant
  least-squares factor `a_p = sum(M*S_p)/sum(M**2)` fitted separately at each piston.
  The neutral value is **1 in both maps**. Wavelength-dependent structure moving with
  piston remains visible after wavelength-independent brightness changes are removed.

All acquired piston samples are included without thinning. Piston rows are sorted
for display only; exports retain acquisition order. The displayed piston is the DM
surface stroke, with **OPD = 2 * piston**. Missing acquisitions, missing spectral pixels,
and nonpositive/low mean-flux columns remain grey. The wavelength calibration and
mean-flux floor match the earlier tools; no extrapolation is applied.

Each panel has its own colourbar centred on 1 by default; enable **Shared scale** for
identical colour limits. Automatic limits cover the full finite displayed range (with
a minimum span of 1 +/- 0.01). Optional manual limits can zoom the contrast; colourbar
arrows indicate clipped values. Optional smoothing acts only along wavelength within
finite runs, never across piston rows or gaps. Brightness factors and exported original
ratios always use unsmoothed data. With only one acquired piston, the mean equals that
spectrum and these maps cannot diagnose piston-dependent changes.


In [ ]:
# PISTON-VERSUS-WAVELENGTH MAP CONTROLS (FIXED FOCUS)
PISTON_WAVELENGTH_SEGMENT = SPECTRA_SEGMENT
PISTON_WAVELENGTH_FOCUS_INDEX = SELECTED_FOCUS_INDEX  # fixed absolute mount-Z focus, selected by index
PISTON_WAVELENGTH_MIN_MEAN_FRACTION = 0.01
PISTON_WAVELENGTH_MIN_MEAN_ADU = 0.0
PISTON_WAVELENGTH_CMAP = "RdBu_r"  # blue < 1, white = 1, red > 1
PISTON_WAVELENGTH_SHARED_SCALE = False
PISTON_WAVELENGTH_RAW_LIMITS = None        # None: automatic; or e.g. (0.8, 1.2), straddling 1
PISTON_WAVELENGTH_CORRECTED_LIMITS = None  # None: automatic; or e.g. (0.95, 1.05)
PISTON_WAVELENGTH_SHARED_LIMITS = None     # used only when Shared scale is enabled
PISTON_WAVELENGTH_SMOOTH_RAW = False
PISTON_WAVELENGTH_SMOOTH_CORRECTED = False
PISTON_WAVELENGTH_SMOOTH_WINDOW = 7  # odd detector-column window; display only
PISTON_WAVELENGTH_SAVE_EXPORTS = False  # PNG, piston/brightness summary CSV, wavelength-resolved ratios CSV


In [ ]:
from matplotlib.colors import TwoSlopeNorm


def compute_piston_wavelength_ratios(data, segment, focus_index, wavelength_nm,
                                     min_mean_fraction=0.01, min_mean_adu=0.0):
    """Use the same across-piston mean and brightness fit as the 1D ratio tool."""
    count = len(data["piston_cmd"])
    base = compute_spectral_ratios(data, segment, focus_index, np.arange(count), wavelength_nm,
                                   mean_scope="all", min_mean_fraction=min_mean_fraction,
                                   min_mean_adu=min_mean_adu)
    raw = np.full((count, len(wavelength_nm)), np.nan)
    corrected = np.full_like(raw, np.nan)
    plotted = base["plotted_indices"]
    raw[plotted] = base["ratios"]
    for row, ip in enumerate(plotted):
        scale = base["summary"].iloc[row]["scale_to_mean"]
        if np.isfinite(scale) and scale > 0:
            corrected[ip] = base["ratios"][row]/scale
    segment, focus_index = int(segment), int(focus_index)
    acquired = np.asarray(data["acquired"][focus_index, segment], dtype=bool)
    summary = pd.DataFrame(dict(piston_index=np.arange(count),
                                piston_um=np.asarray(data["piston_cmd"], dtype=float),
                                opd_um=2*np.asarray(data["piston_cmd"], dtype=float), acquired=acquired))
    metrics = base["summary"].drop(columns=["piston_um", "opd_um"]).rename(
        columns={"scale_to_mean":"scale_to_piston_mean"})
    summary = summary.merge(metrics, on="piston_index", how="left", sort=False)
    return dict(raw_ratios=raw, corrected_ratios=corrected, mean_spectrum=base["mean_spectrum"],
                reference_count=base["reference_count"], valid_column=base["valid_column"], acquired=acquired,
                summary=summary, floor_adu=base["floor_adu"], wavelength_nm=np.asarray(wavelength_nm),
                segment=segment, focus_index=focus_index)


def piston_wavelength_colour_limits(values, limits=None):
    if limits is not None:
        bounds = np.asarray(limits, dtype=float)
        if bounds.shape != (2,) or not np.all(np.isfinite(bounds)) or not bounds[0] < 1 < bounds[1]:
            raise ValueError("Colour limits must be finite and straddle 1")
        return tuple(bounds)
    finite = np.asarray(values)[np.isfinite(values)]
    half_range = max(float(np.max(np.abs(finite-1))) if finite.size else 0., 0.01)
    return 1-half_range, 1+half_range


def plot_piston_wavelength_maps(segment=PISTON_WAVELENGTH_SEGMENT, focus_index=PISTON_WAVELENGTH_FOCUS_INDEX,
                             shared_scale=PISTON_WAVELENGTH_SHARED_SCALE,
                             smooth_raw=PISTON_WAVELENGTH_SMOOTH_RAW,
                             smooth_corrected=PISTON_WAVELENGTH_SMOOTH_CORRECTED,
                             smooth_window=PISTON_WAVELENGTH_SMOOTH_WINDOW,
                             min_mean_fraction=PISTON_WAVELENGTH_MIN_MEAN_FRACTION,
                             min_mean_adu=PISTON_WAVELENGTH_MIN_MEAN_ADU,
                             raw_limits=PISTON_WAVELENGTH_RAW_LIMITS,
                             corrected_limits=PISTON_WAVELENGTH_CORRECTED_LIMITS,
                             shared_limits=PISTON_WAVELENGTH_SHARED_LIMITS, save_exports=False):
    result = compute_piston_wavelength_ratios(scan, segment, focus_index, spectra_wavelength_nm,
                                          min_mean_fraction, min_mean_adu)
    raw_display = smooth_ratio_display(result["raw_ratios"], smooth_raw, smooth_window)
    corrected_display = smooth_ratio_display(result["corrected_ratios"], smooth_corrected, smooth_window)
    result.update(display_raw_ratios=raw_display, display_corrected_ratios=corrected_display,
                  smoothing=dict(raw=bool(smooth_raw), corrected=bool(smooth_corrected),
                                  window_columns=int(smooth_window) if smooth_raw or smooth_corrected else None))
    if shared_scale:
        common = piston_wavelength_colour_limits(np.r_[raw_display.ravel(), corrected_display.ravel()], shared_limits)
        ranges = [common, common]
    else:
        ranges = [piston_wavelength_colour_limits(raw_display, raw_limits),
                  piston_wavelength_colour_limits(corrected_display, corrected_limits)]
    piston_order = np.argsort(scan["piston_cmd"])
    x_order = np.flatnonzero(np.isfinite(spectra_wavelength_nm))
    x_order = x_order[np.argsort(spectra_wavelength_nm[x_order])]
    if not len(x_order):
        raise ValueError("No calibrated spectral columns")
    pistons = np.asarray(scan["piston_cmd"])[piston_order]
    wavelength = spectra_wavelength_nm[x_order]
    if (len(pistons) > 1 and not np.all(np.diff(pistons) > 0)) or (len(wavelength) > 1 and not np.all(np.diff(wavelength) > 0)):
        raise ValueError("Piston and wavelength coordinates must be unique")
    with plt.ioff():  # suppress backend auto-display before the explicit output
        fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
    cmap = plt.get_cmap(PISTON_WAVELENGTH_CMAP).copy()
    cmap.set_bad("lightgrey")
    titles = ["Raw: spectrum / mean across piston", "Brightness corrected: spectrum / (a * mean)"]
    result["colour_limits"] = dict(raw=ranges[0], corrected=ranges[1])
    result["actual_ranges"] = {}
    for panel, (ax, values, limits, title) in enumerate(zip(axes, [raw_display, corrected_display], ranges, titles)):
        image = ax.pcolormesh(cell_edges(wavelength), cell_edges(pistons, .05),
                               np.ma.masked_invalid(values[np.ix_(piston_order, x_order)]),
                               norm=TwoSlopeNorm(vmin=limits[0], vcenter=1., vmax=limits[1]),
                               cmap=cmap, shading="flat", rasterized=True)
        finite = values[np.isfinite(values)]
        below = bool(finite.size and finite.min() < limits[0])
        above = bool(finite.size and finite.max() > limits[1])
        extend = "both" if below and above else "min" if below else "max" if above else "neither"
        key = "raw" if panel == 0 else "corrected"
        result["actual_ranges"][key] = (float(finite.min()), float(finite.max())) if finite.size else (np.nan, np.nan)
        fig.colorbar(image, ax=ax, label="Spectral ratio (neutral = 1)", extend=extend)
        ax.set(xlabel="Wavelength [nm]", title=title)
        if not finite.size:
            ax.text(.5, .5, "No valid ratios for this selection", transform=ax.transAxes,
                    ha="center", va="center", bbox=dict(facecolor="white", alpha=.8))
    axes[0].set_ylabel("DM surface piston [um]")
    fixed_z = float(scan["focus_cmd"][int(focus_index)])
    fig.suptitle(f"DM segment {segment} | fixed focus #{int(focus_index)}: Z={fixed_z:g} pulses", fontsize=13)
    smoothing_text = (f"Display smoothing raw={'on' if smooth_raw else 'off'}, corrected={'on' if smooth_corrected else 'off'}"
                      + (f" ({int(smooth_window)} columns)" if smooth_raw or smooth_corrected else ""))
    nreference = np.count_nonzero(result["acquired"] & np.any(np.isfinite(result["raw_ratios"]), axis=1))
    fig.text(.08, .025, f"{source_label}\nMean across {nreference} valid acquired piston samples at this fixed focus; "
             f"grey = missing/masked. Colour scales {'shared' if shared_scale else 'independent'}.\n"
             f"{smoothing_text}; brightness factors use unsmoothed data.", fontsize=8)
    fig.subplots_adjust(left=.08, right=.97, bottom=.20, top=.84, wspace=.25)
    result["saved_files"] = []
    if save_exports:
        suffix = (f"_raw{int(bool(smooth_raw))}_corrected{int(bool(smooth_corrected))}_w{int(smooth_window)}"
                  if smooth_raw or smooth_corrected else "")
        stem = output_dir/f"piston_wavelength_segment_{int(segment):02d}_focus_{int(focus_index):03d}{suffix}"
        png = stem.with_suffix(".png")
        fig.savefig(png, dpi=160, facecolor="white")
        summary_path = str(stem)+"_summary.csv"
        result["summary"].to_csv(summary_path, index=False)
        npiston, nx = result["raw_ratios"].shape
        table = pd.DataFrame(dict(focus_index=int(focus_index), focus_z_pulses=fixed_z,
                                   piston_index=np.repeat(np.arange(npiston), nx),
                                   piston_um=np.repeat(scan["piston_cmd"], nx),
                                   opd_um=np.repeat(2*np.asarray(scan["piston_cmd"]), nx),
                                   wavelength_nm=np.tile(spectra_wavelength_nm, npiston),
                                   acquired=np.repeat(result["acquired"], nx),
                                   piston_mean_spectrum=np.tile(result["mean_spectrum"], npiston),
                                   raw_ratio=result["raw_ratios"].ravel(),
                                   corrected_ratio=result["corrected_ratios"].ravel(),
                                   display_raw_ratio=raw_display.ravel(), display_corrected_ratio=corrected_display.ravel()))
        csv = str(stem)+"_ratios.csv"
        table.to_csv(csv, index=False)
        result["saved_files"] = [str(png), summary_path, csv]
    return fig, result


def show_piston_wavelength_maps(**selection):
    fig, result = plot_piston_wavelength_maps(**selection)
    display_analysis_figure(fig)
    print("Displayed data min/max:", result["actual_ranges"], "| colourbar limits:", result["colour_limits"])
    display(result["summary"])
    for path in result["saved_files"]:
        print("Saved:", path)
    return result


# Match the existing tools: one widget output OR one static figure, never both.
for _widget_name in ("focus_spectral_ui", "focus_spectral_output", "piston_wavelength_ui", "piston_wavelength_output"):
    _previous_widget = globals().get(_widget_name)
    if _previous_widget is not None:
        _previous_widget.close()
_piston_wavelength_widgets_available = False
if USE_INTERACTIVE_WIDGETS:
    try:
        import ipywidgets as piston_wavelength_widgets
    except ImportError:
        print("ipywidgets unavailable; displaying one static piston/wavelength map.")
    else:
        _piston_wavelength_widgets_available = True
if _piston_wavelength_widgets_available:
    piston_wavelength_controls = dict(
        segment=piston_wavelength_widgets.Dropdown(options=list(range(37)), value=PISTON_WAVELENGTH_SEGMENT, description="Segment"),
        focus_index=piston_wavelength_widgets.Dropdown(
            options=[(f"#{iz}: Z={z:g} pulses", iz) for iz,z in enumerate(scan["focus_cmd"])],
            value=PISTON_WAVELENGTH_FOCUS_INDEX, description="Fixed focus", layout={"width":"300px"}),
        shared_scale=piston_wavelength_widgets.Checkbox(value=PISTON_WAVELENGTH_SHARED_SCALE, description="Shared scale"),
        smooth_raw=piston_wavelength_widgets.Checkbox(value=PISTON_WAVELENGTH_SMOOTH_RAW, description="Smooth raw"),
        smooth_corrected=piston_wavelength_widgets.Checkbox(value=PISTON_WAVELENGTH_SMOOTH_CORRECTED, description="Smooth corrected"),
        smooth_window=piston_wavelength_widgets.IntSlider(value=PISTON_WAVELENGTH_SMOOTH_WINDOW, min=1,
            max=max(31, PISTON_WAVELENGTH_SMOOTH_WINDOW), step=2, description="Window [cols]", continuous_update=False),
        save_exports=piston_wavelength_widgets.Checkbox(value=PISTON_WAVELENGTH_SAVE_EXPORTS, description="Save PNG/CSV"))
    def _update_piston_wavelength_maps(**selection):
        global piston_wavelength_map_result
        piston_wavelength_map_result = show_piston_wavelength_maps(**selection)
    piston_wavelength_output = piston_wavelength_widgets.interactive_output(_update_piston_wavelength_maps, piston_wavelength_controls)
    piston_wavelength_ui = piston_wavelength_widgets.VBox([
        piston_wavelength_widgets.HTML("<b>Fixed focus: dispersed spectral ratios across piston</b>"),
        piston_wavelength_widgets.HBox([piston_wavelength_controls["segment"], piston_wavelength_controls["focus_index"],
                                      piston_wavelength_controls["shared_scale"]]),
        piston_wavelength_widgets.HBox([piston_wavelength_controls["smooth_raw"], piston_wavelength_controls["smooth_corrected"],
                                      piston_wavelength_controls["smooth_window"], piston_wavelength_controls["save_exports"]]),
        piston_wavelength_output])
    display(piston_wavelength_ui)
else:
    piston_wavelength_map_result = show_piston_wavelength_maps(
        segment=PISTON_WAVELENGTH_SEGMENT, focus_index=PISTON_WAVELENGTH_FOCUS_INDEX,
        shared_scale=PISTON_WAVELENGTH_SHARED_SCALE, smooth_raw=PISTON_WAVELENGTH_SMOOTH_RAW,
        smooth_corrected=PISTON_WAVELENGTH_SMOOTH_CORRECTED, smooth_window=PISTON_WAVELENGTH_SMOOTH_WINDOW,
        save_exports=PISTON_WAVELENGTH_SAVE_EXPORTS)
